In [ ]:
import os
os.environ["TORCH_DYNAMO_DISABLE"] = "1"                          
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"                                             

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
import multiprocessing

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import jax
import jax.numpy as jnp
import jax.nn as jnn


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# =================================================================
                                
# =================================================================
class FourierFeatureMapping(nn.Module):
    def __init__(self, in_dims=2, mapping_size=128, scale=10.0):
        super().__init__()
                                  
        self.register_buffer('B', torch.randn(in_dims, mapping_size) * scale)

    def forward(self, x):
        # x: (Batch, 2)
        # x @ B: (Batch, mapping_size)
        x_proj = (2.0 * np.pi * x) @ self.B
                                          
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)

# =================================================================
               
# =================================================================
class DenseResBlock(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.fc1 = nn.Linear(size, size)
        self.fc2 = nn.Linear(size, size)
        self.norm1 = nn.LayerNorm(size)
        self.norm2 = nn.LayerNorm(size)

    def forward(self, x):
        res = x
        x = self.norm1(x)
        x = F.silu(self.fc1(x))
        x = self.norm2(x)
        x = F.silu(self.fc2(x))
        return x + res

# =================================================================
                       
# =================================================================
class HighFidelityNeuralVF(nn.Module):
    def __init__(self, n_poles=12, width=256, fourier_size=128,num=6):
        super().__init__()
        self.n_poles = n_poles
        
                                         
        self.fourier_map = FourierFeatureMapping(in_dims=2, mapping_size=fourier_size, scale=5.0)
        
                                 
        in_features = fourier_size * 2 + 2 
        self.in_layer = nn.Linear(in_features, width)
        
                           
        self.res_layers = nn.ModuleList([DenseResBlock(width) for _ in range(num)])
        
                
        self.out_layer = nn.Linear(width, 2 * n_poles)

    def forward(self, x):
        # x shape: (Batch, 2)
        
                
        x_fourier = self.fourier_map(x)
                                            
        x_concat = torch.cat([x, x_fourier], dim=-1)
        
        h = F.silu(self.in_layer(x_concat))
        for layer in self.res_layers:
            h = layer(h)
        out = self.out_layer(h)
        
                             
        log_tau_k = out[:, 0 : self.n_poles]
        log_w_k = out[:, self.n_poles :]
        
        tau_k = 10.0 ** log_tau_k
        w_k = 10.0 ** log_w_k
                                  
                                        
        w_k_sum = torch.sum(w_k, dim=1, keepdim=True) + 1e-12        
        w_k = w_k / w_k_sum
        
        return tau_k, w_k, log_tau_k, log_w_k

In [ ]:

# =================================================================
              
# =================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HighFidelityNeuralVF().to(device)




In [ ]:
                 
                              
                                      
model.load_state_dict(torch.load("model_trained.pth"))
model.eval()
print("[*] 模型已从 model_trained.pth 加载并进入推理模式")

In [ ]:

                         
torch.manual_seed(42)

numpy_weights = {}
for name, param in model.state_dict().items():
    numpy_weights[name] = param.detach().cpu().numpy()


In [ ]:

class PureJAXIPNet:
    """完全使用 jax.numpy 重写的高精度物理神经网络映射器"""
    
    def __init__(self, params_dict, n_poles=12, dtype=jnp.float32):
                                                                 
        self.params = jax.tree.map(lambda x: jnp.asarray(x, dtype=dtype), params_dict)
        self.n_poles = int(n_poles)

        # =========================================================
                                               
        # =========================================================
        @jax.jit
        def _forward_fn(params, x):
            
            def _fourier_map(x_in):
                B = params['fourier_map.B']
                x_proj = (2.0 * jnp.pi * x_in) @ B
                return jnp.concatenate([jnp.sin(x_proj), jnp.cos(x_proj)], axis=-1)

            def _layer_norm(x_in, weight, bias, eps=1e-5):
                mean = jnp.mean(x_in, axis=-1, keepdims=True)
                var = jnp.var(x_in, axis=-1, keepdims=True)
                return (x_in - mean) / jnp.sqrt(var + eps) * weight + bias

            def _res_block(x_in, prefix):
                fc1_w, fc1_b = params[f'{prefix}.fc1.weight'], params[f'{prefix}.fc1.bias']
                fc2_w, fc2_b = params[f'{prefix}.fc2.weight'], params[f'{prefix}.fc2.bias']
                ln1_w, ln1_b = params[f'{prefix}.norm1.weight'], params[f'{prefix}.norm1.bias']
                ln2_w, ln2_b = params[f'{prefix}.norm2.weight'], params[f'{prefix}.norm2.bias']

                res = x_in
                x_tmp = _layer_norm(x_in, ln1_w, ln1_b)
                x_tmp = jnn.silu(x_tmp @ fc1_w.T + fc1_b)
                
                x_tmp = _layer_norm(x_tmp, ln2_w, ln2_b)
                x_tmp = jnn.silu(x_tmp @ fc2_w.T + fc2_b)
                return x_tmp + res

            x_fourier = _fourier_map(x)
            x_concat = jnp.concatenate([x, x_fourier], axis=-1)

            in_w, in_b = params['in_layer.weight'], params['in_layer.bias']
            h = jnn.silu(x_concat @ in_w.T + in_b)

                         
            for i in range(6):
                h = _res_block(h, f'res_layers.{i}')

            out_w, out_b = params['out_layer.weight'], params['out_layer.bias']
            out = h @ out_w.T + out_b

                                                 
            log_tau_k = out[:, 0 : self.n_poles]
            log_w_k = out[:, self.n_poles :]
            
            tau_k = 10.0 ** log_tau_k
            w_k = 10.0 ** log_w_k
                                 
                                            
            w_k_sum = jnp.sum(w_k, axis=1, keepdims=True) + 1e-12        
       
            w_k = w_k / w_k_sum 
       
        
            return  tau_k, w_k

                        
        self._jit_forward = _forward_fn

    def __call__(self, x):
                                                       
        return self._jit_forward(self.params, x)



In [ ]:

# =================================================================
                                  
# =================================================================

                            
dummy_input = np.array([
    [0.5, -2.0],
    [0.2, -1.0],
    [0.8, -3.0]
], dtype=np.float32)

print("[-] 正在运行 PyTorch 基准测试...")

                                            
                                                        
device = next(model.parameters()).device 

                                         
torch_input = torch.from_numpy(dummy_input).to(device)

with torch.no_grad():
    pt_tau_k, pt_w_k, _, _ = model(torch_input)

print("[-] 正在运行 JAX (XLA 编译) 推理测试...")
jax_model = PureJAXIPNet(numpy_weights)                   



jax_input = jnp.asarray(dummy_input)
                 
jax_tau_k, jax_w_k = jax_model(jax_input)


In [ ]:
def save_jax_model(model, filepath):
    """
    保存 PureJAXIPNet 模型权重
    """
                                               
                                         
    numpy_params = jax.tree.map(lambda x: np.asarray(x), model.params)
    
                                                             
    np.savez_compressed(filepath, **numpy_params)
    print(f"✅ 模型参数已成功压缩保存至: {filepath}")

                        
save_jax_model(jax_model, f"jax_ip_model_{12}poles.npz")

In [ ]:
import numpy as np

def load_jax_model(filepath, n_poles=18):
    """
    从硬盘加载权重并重建 PureJAXIPNet 模型
    """
               
    loaded_data = np.load(filepath)
    
                                             
    loaded_params = {key: loaded_data[key] for key in loaded_data.files}
    
                          
    new_model = PureJAXIPNet(loaded_params, n_poles=n_poles)
    print(f"✅ 成功从 {filepath} 加载模型并初始化 JAX 引擎！")
    
    return new_model

      
loaded_jax_model = load_jax_model(f"jax_ip_model_{12}poles.npz", n_poles=10)

               
dummy_input = jnp.array([[0.5, -2.0], [0.2, -1.0]])
tau, w = loaded_jax_model(dummy_input)
print("测试输出 tau 形状:", tau.shape)